## 인코더

- 입력을 읽어서 흔적을 남기는 것
- 벡터로 바꾼다. (Embedding)

In [5]:
import torch
import torch.nn as nn

In [6]:
torch.__version__

'2.13.0+cu130'

In [8]:
class Encorder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()  # 상속 부모의 생성자 호출
        self.emb = nn.Embedding    # 임베딩
        self.gru = nn.GRU(32, 64, batch_first=True) # GRU
    
    def forward(self, x):
        return self.gru(self.emb(x))[0]

## 어텐션

In [ ]:
class Attention(nn.Module):
    def __init__(self):
        super().__init__()
        self.Wq = nn.Linear(64, 64)         # q 가중치, 디코더 상태변환
        self.Wk = nn.Linear(2 * 64, 64)     # 인코더 hidden 변환
        self.v = nn.Linear(64, 1)           # 점수
    
    def forward(self, dec_h, enc_out):
        q = self.Wq(dec_h).unsqueeze(1)     # (B, 1, H)
        k = self.Wk(enc_out)                # (B, S, H)
        score = self.v(torch.tanh(q + k)).squeeze(-1)   # (B, S)
        weights = torch.softmax(score, dim=1)   # 합이 1인 가중치 (B, S)
        context = (weights.unsqueeze(-1) * enc_out).sum(1)  # 가중합
        return context, weights

## 디코더

In [11]:
class Decorder(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()  
        self.emb = nn.Embedding(vocab_size, 32) #임베딩
        self.attn = Attention()                 # 어텐션
        self.cell = nn.GRUCell(32 + 2 *64, 64)  # GRU 한글자씩 ->  GRUCell
        self.out = nn.Linear(64, vocab_size)    # 출력층
            
    def forward(self, y_prev, enc_out, h, uniform_len=None):
        # uniform_len == None 어텐션 끈다.
        if uniform_len is None:
            context, weights = self.attn(h, enc_out)
        else:
            weights = torch.zeros(enc_out.size(0), enc_out.size(1))
            weights[:, :uniform_len] = 1.0 / uniform_len
            context = (weights.unsqueeze(-1) * enc_out).sum(1)
        h = self.cell(torch.cat([self.emb(y_prev), context], dim=1), h)
        return self.out(h), h, weights